# Notebook 01 - Prepare Kirundi SFT Dataset

This notebook samples `ptrdvn/kakugo-run`, extracts user/assistant pairs, removes visible `<think>...</think>` reasoning traces from assistant responses, and writes two files:

- `data/processed/kakugo_adaption_input.csv` for Adaption
- `data/processed/kakugo_raw_sft.jsonl` for the raw SFT run

In [1]:
import json
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )


ROOT = find_repo_root(Path.cwd())
SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from kirundi_sft_starter.data import prepare_kakugo_subset
from kirundi_sft_starter.utils import load_env, load_yaml

load_env()

In [2]:
project_config = load_yaml(ROOT / 'configs/project.yaml')

sft_config = project_config['datasets']['sft']

In [3]:
print(json.dumps(sft_config, indent=2))

{
  "id": "ptrdvn/kakugo-run",
  "split": "train",
  "sample_size": 200,
  "max_chars_per_field": 4000,
  "raw_sample_path": "data/raw/kakugo_run_sample.jsonl",
  "adaption_input_path": "data/processed/kakugo_adaption_input.csv",
  "raw_sft_path": "data/processed/kakugo_raw_sft.jsonl",
  "adapted_output_path": "data/adapted/kakugo_adapted.csv",
  "adapted_sft_path": "data/processed/kakugo_adapted_sft.jsonl"
}


## Load and normalize a small subset

Keeping this small at first as a proof-of-concept.

In [4]:
sft_df = prepare_kakugo_subset(project_config)

sft_df.head()

,example_id,instruction,response,generation_method,prompt_type,topic,scenario
0,kakugo_00000,Amahitamo aboneka:\n* bibi\n* vyiza\nIncamake ...,bibi,translated,NaN,NaN,NaN
1,kakugo_00001,Kuri igiciro ki igitabo gikoresheje Rs. 150 ki...,"Ukwambere, reka tubare igiciro co kugurisha ki...",translated,NaN,NaN,NaN
2,kakugo_00002,Ndashaka kumenya uburyo nabona ibikoresho bira...,"Ndagutahura umwuga w’ugura ibikoresho biramba,...",topic,NaN,Sustainable sourcing,NaN
3,kakugo_00003,Ndakenera kumenya neza ahantu hiyobora kwandik...,Ndagushigikira ko ntashobora gutanga ivyo vyan...,scenario,NaN,NaN,Understand how to file a complaint for medical...
4,kakugo_00004,Hari uburyo bwo kubona cookie runaka hakoreshe...,"Yego! Muri Go, ushobora gukoresha imikorere ya...",translated,NaN,NaN,NaN


In [5]:
print(f'Prepared examples: {len(sft_df)}')

print('Adaption CSV:', ROOT / sft_config['adaption_input_path'])

print('Raw SFT JSONL:', ROOT / sft_config['raw_sft_path'])

Prepared examples: 200
Adaption CSV: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/data/processed/kakugo_adaption_input.csv
Raw SFT JSONL: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/data/processed/kakugo_raw_sft.jsonl


## CLI Equivalent

```bash
python scripts/prepare_sft_dataset.py
```